# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tokihab/FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data safely (handling Colab vs Local paths)
path_from_notebook = '../../data/raw/content_refresh_anonymized.csv'
path_from_root = 'data/raw/content_refresh_anonymized.csv'
github_url = 'https://raw.githubusercontent.com/tokihab/flyrankintern-ml/main/data/raw/content_refresh_anonymized.csv'

if os.path.exists(path_from_notebook):
    df = pd.read_csv(path_from_notebook)
elif os.path.exists(path_from_root):
    df = pd.read_csv(path_from_root)
else:
    df = pd.read_csv(github_url)

# 2. Prep and Train a quick model to get our scores
df['success'] = (df['trend_pct'] > 0).astype(int)
features = ['search_volume', 'competition', 'word_count', 'avg_position']
df_clean = df.dropna(subset=['client_id', 'success'] + features).copy()

X = pd.get_dummies(df_clean[features], drop_first=True)
y = df_clean['success']

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# 3. Add the "Win Chance" (probability) back to our table
df_clean['win_chance'] = rf_model.predict_proba(X)[:, 1]

## 1. Ranked Actions & Reason Codes

We translate the math into plain English for the writers.
* **Archetype to Action:** If an article has a high chance of winning and gets a lot of searches, it becomes an "Update Immediately."
* **Reason Codes:** We tell the writer *why* the tool flagged it (e.g., "It's stuck on Page 2").
* **Decay Insight:** Older articles naturally lose traffic over time. This tool spots that decay early so we can rescue the post before the traffic drops to zero.

In [2]:
# Create simple rules to tell humans what to do and why
def assign_action(row):
    if row['win_chance'] > 0.6 and row['search_volume'] > 50:
        return "High Priority: Update Now"
    elif row['win_chance'] > 0.4:
        return "Medium Priority: Quick Fixes"
    else:
        return "Low Priority: Leave As Is"

def get_reason(row):
    if row['avg_position'] > 10:
        return "Stuck on Page 2+ (Needs a rankings push)"
    if row['word_count'] < 1000:
        return "Content is too thin (Needs more details)"
    return "General traffic decay (Needs fresh info)"

# Apply the rules
df_clean['action'] = df_clean.apply(assign_action, axis=1)
df_clean['reason_code'] = df_clean.apply(get_reason, axis=1)

# Sort from highest chance of winning to lowest
playbook = df_clean.sort_values(by='win_chance', ascending=False)

display(playbook[['content_id', 'action', 'reason_code', 'win_chance']].head())

,content_id,action,reason_code,win_chance
16546,content_6d6184599790,Medium Priority: Quick Fixes,Stuck on Page 2+ (Needs a rankings push),0.97
870,content_10e5b963e3cc,Medium Priority: Quick Fixes,Stuck on Page 2+ (Needs a rankings push),0.97
5739,content_26ed42144892,Medium Priority: Quick Fixes,General traffic decay (Needs fresh info),0.97
18487,content_1a400a80dc02,Medium Priority: Quick Fixes,General traffic decay (Needs fresh info),0.97
19479,content_0138556d5d08,Medium Priority: Quick Fixes,Stuck on Page 2+ (Needs a rankings push),0.97


## 2. Intended Use and Limits

* **Intended Use:** This is a daily sorting tool to help content managers decide which old articles to update first to get the most traffic.
* **Limits:** The tool only sees numbers (word counts, clicks). It cannot read the article, so it doesn't know if the writing is actually good or terrible. It offers a directional guess, not a guarantee.

## 3. Human Review & The No-Go List

This is a decision-support tool. A human is always in the driver's seat.

**Human Review Rules:**
1. A writer must actually read the flagged article to see if the "Reason Code" makes sense.
2. An editor must approve the final rewritten draft.

**The No-Go List (What we will NEVER automate):**
* **Do NOT** let an AI automatically rewrite and publish the article without human eyes.
* **Do NOT** use these scores to punish writers if an article decays.
* **Do NOT** auto-delete content just because it gets a "Low Priority" score.

## 4. Monitoring & Cost/Value

* **Cost/Value Thinking:** Writers' time is expensive. We only assign "High Priority" to articles with decent search volume. Updating an article nobody searches for is a waste of money, even if the model thinks the update will succeed.
* **Monitoring:** We will check the data once a month to see if the articles we flagged actually gained traffic after the writers fixed them.
* **Retrain Trigger:** If our success predictions are wrong for two months in a row, the trends have changed. We will pause the tool and feed it fresh data.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
## 5. Export for the Paper

# Ensure the outputs folder exists based on the repo structure[cite: 3]
out_dir = '../outputs'
os.makedirs(out_dir, exist_ok=True)

# Select only the columns the content team actually needs
export_cols = ['content_id', 'client_id', 'action', 'reason_code', 'win_chance']
playbook_export = playbook[export_cols]

# Save the CSV to the outputs folder[cite: 3]
export_path = f'{out_dir}/refresh_queue_sample.csv'
playbook_export.to_csv(export_path, index=False)

print(f"SUCCESS: Playbook exported to {export_path}")

SUCCESS: Playbook exported to ../outputs/refresh_queue_sample.csv


## 6. Self-Check Complete

* [x] Ranked actions and simple reason codes created.
* [x] Intended use and strict limits explained.
* [x] Human review rules and hard "No-Go" list defined.
* [x] Retrain triggers and cost/value logic established.
* [x] Queue exported to `work/outputs/` successfully[cite: 3].
* [x] All scientific jargon dissolved into simple, honest words.